# Trabajo Práctico: Algoritmos de Optimización

**Integrantes - Grupo 16:**
* **Alumno 1:** Nathalie Alejandra Véliz Godoy
* **Alumno 2:** Constanza de los Angeles Solá
* **Google Colab:**  https://drive.google.com/drive/folders/1gCTXHFGsoWIOo3FiAV82N5rkYmdEGa7d?usp=drive_link
* **Github** https://github.com/contisola/master_ia_viu_26/

**Asignatura:** Algoritmos de Optimización — Máster en Inteligencia Artificial (VIU)  
**Problema elegido:** Problema 1 — Organizar sesiones de doblaje
                       

## Enunciado del problema

Se precisa coordinar el doblaje de una película. Los actores de doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos. Los actores cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación, **independientemente del número de tomas que graben ese día**. **No es posible grabar más de 6 tomas por día.** El objetivo es planificar las sesiones por día de manera que el gasto total por los servicios de los actores sea el menor posible.

- **Número de actores:** 10
- **Número de tomas:** 30
- **Datos de entrada:** una matriz de 30×10 donde un 1 indica que el actor participa en la toma y un 0 que no.
- **Actores/Tomas:** https://bit.ly/36D8IuK
- **Restricción estricta:** máximo 6 tomas por día.

Este cuaderno desarrolla el problema en cuatro bloques: **Fase 1 · Modelo del problema**, **Fase 2 · Diseño y análisis del algoritmo por fuerza bruta**, **Fase 3 · Diseño y análisis del algoritmo mejorado (Recocido Simulado)** y **Fase 4 · Validación empírica con datos aleatorios**, seguido de las referencias y las líneas de trabajo futuro. Las respuestas obligatorias se marcan con **(\*)**.

---
## FASE 1 · Modelo del problema

### Pregunta 1 (*) — ¿Cuántas posibilidades hay sin tener en cuenta las restricciones?<br>

Sin tener en cuenta las restricciones del problema, podríamos agendar las 30 tomas en hasta 30 días distintos (asignando una toma por día). Como cada una de las 30 tomas puede asignarse de forma totalmente independiente a cualquiera de esos 30 días, estamos ante una **Variación con Repetición**.

La fórmula general de la Variación con Repetición es:
$$VR_{n,k} = n^k$$

Donde $n = 30$ (días posibles) y $k = 30$ (tomas por agendar). El cálculo del espacio total de búsqueda ($|\Omega|$) es:

$$|\Omega| = 30^{30}$$

Esto nos da un total astronómico de aproximadamente **$2.06 \times 10^{44}$** posibilidades.

<br>

### Pregunta 2 — ¿Cuántas posibilidades hay teniendo en cuenta todas las restricciones?

La restricción estricta es un máximo de 6 tomas por día. Para agendar 30 tomas, el mínimo de días estrictamente necesarios es:

$$D_{min} = \frac{30}{6} = 5 \text{ días}$$

Por el Principio del Palomar, la única forma de acomodar 30 tomas en 5 días sin superar el límite de 6, es que **los 5 días tengan exactamente 6 tomas cada uno**. 

Para calcular esto, usamos una **Permutación con Repetición**, donde tomamos el total de ordenamientos y dividimos para eliminar los "clones", ya que el orden interno de grabación de las 6 tomas en un mismo día no altera el resultado final ni crea combinaciones nuevas.

La fórmula general de la Permutación con Repetición es:
$$PR = \frac{n!}{n_1! \cdot n_2! \dots n_m!}$$

Donde $n = 30$ (total de tomas) y tenemos 5 grupos idénticos de 6 tomas cada uno. Aplicando la fórmula, obtenemos las asignaciones válidas ($|\Omega_{validas}|$):

$$|\Omega_{validas}| = \frac{30!}{6! \cdot 6! \cdot 6! \cdot 6! \cdot 6!} = \frac{30!}{(6!)^5}$$

Esto reduce nuestro espacio de búsqueda a unas **$1.37 \times 10^{18}$** asignaciones válidas.


> **Nota:** este cálculo asume el escenario de mínimo número de días (5), es decir, la forma más compacta de repartir las tomas respetando la restricción de 6 por día. Si se permitieran más de 5 días (dejando algún día con menos de 6 tomas), el número de asignaciones válidas sería aún mayor. Tomamos este caso como referencia porque representa la cota inferior del espacio de búsqueda factible y es el punto de partida que usamos para dimensionar el problema.

### Pregunta 3 (*) — ¿Cuál es la estructura de datos que mejor se adapta al problema? 

Para modelar el espacio de soluciones de este problema, una aproximación altamente eficiente es utilizar una combinación de un **Vector de Asignación** (como estructura principal) y **Máscaras de Bits** (como estructura auxiliar).

#### 1. Estructura Principal: Vector de Asignación (Arreglo Unidimensional)
El calendario se modela como un arreglo plano de tamaño fijo, donde la longitud es igual al número total de tomas ($n = 30$). 
*   **Formato:**
$$
S = [d_0, d_2, \dots, d_{29}]
$$
*   **Lógica:** El índice del arreglo representa el ID de la toma, y el valor almacenado en esa posición representa el día asignado para su grabación.

**Argumento de elección:**
Inicialmente, se podría pensar en usar una matriz bidimensional o una lista de listas (ej. `Dias = [[toma1, toma2], [toma3], ...]`). Sin embargo, el vector plano es infinitamente superior porque:
*   **Complejidad Espacial:** Mantiene un *footprint* de memoria mínimo y estricto de $O(n)$.
*   **Complejidad Temporal:** Permite consultar o reasignar el día de cualquier toma con acceso directo a memoria en tiempo constante $O(1)$. Para algoritmos metaheurísticos que evalúan miles de mutaciones por segundo, evitar la sobrecarga de buscar elementos dentro de sublistas es crítico para el rendimiento.

#### 2. Estructura Auxiliar: Máscaras de Bits (Bitmasks)
La matriz original de tomas y actores es bidimensional (30 tomas x 10 actores). En lugar de mantenerla así, se comprime la participación de los actores de cada toma en un único número entero utilizando operaciones de desplazamiento de bits (`<<`).

**Argumento de elección:**
Al agrupar las tomas en un día, necesitamos saber cuántos actores distintos asisten. Si usáramos listas tradicionales, tendríamos que iterar sobre cada actor y usar estructuras como `Set()` para eliminar duplicados, lo cual es computacionalmente costoso. 
Al usar máscaras de bits, la unión de todos los actores que participan en un día se resuelve con una simple operación lógica `OR` a nivel de bits (`|`). Las operaciones bit a bit se ejecutan directamente en los registros del procesador, reduciendo drásticamente las "operaciones simples" y acelerando el cálculo de la Función Objetivo de manera masiva.

### Pregunta 4 (*) — ¿Cuál es la función objetivo?
La función objetivo de este modelo calcula el **coste operativo total de las sesiones de grabación**. Este costo se define como el número total de desplazamientos (o jornadas de trabajo) de los actores. 

Matemáticamente, para una solución o calendario $S$, el costo se calcula sumando la cantidad de actores distintos que deben asistir al estudio en cada uno de los días programados. Para mantener la coherencia con los arreglos en Python (cuyos índices inician en 0), la ecuación se expresa de la siguiente manera:

$$f(S) = \sum_{d=0}^{D-1} \left| \bigcup_{i: S[i] = d} A(i) \right|$$

**Desglose de los términos:**
*   $D$: Es el número total de días utilizados en el calendario $S$. Al empezar el índice en 0, la sumatoria recorre desde el día $0$ hasta el día $D-1$.
*   $S[i] = d$: Indica que la toma $i$ (donde $i$ va de $0$ a $29$) ha sido asignada al día $d$.
*   $A(i)$: Representa el conjunto de actores que participan en la toma $i$.
*   $\bigcup$: El operador de unión agrupa a todos los actores requeridos para el día $d$. Al ser un conjunto, elimina automáticamente los duplicados (si un actor graba tres tomas el mismo día, su desplazamiento se cuenta una sola vez).
*   $| \dots |$: Representa la cardinalidad de ese conjunto, es decir, la cantidad exacta de actores distintos que terminaron yendo ese día al estudio.

### Pregunta 5 (*) — ¿Es un problema de maximización o minimización?

Se trata de un problema estricto de **minimización**. 

El propósito del algoritmo es encontrar la combinación (asignación de tomas a días) que arroje el valor más bajo posible en la función objetivo $f(S)$. Es decir, se busca reducir al máximo el número de veces que los actores tienen que viajar al estudio para optimizar el presupuesto, respetando en todo momento la restricción dura de no asignar más de 6 tomas por día.

---
## FASE 2 · Diseño y análisis: Fuerza Bruta

### Pregunta 6 — Diseña un algoritmo para resolver el problema por fuerza bruta (Backtracking con Poda)

Dado que el espacio de soluciones total para las 30 tomas originales es computacionalmente inmanejable ($1.37 \times 10^{18}$ combinaciones válidas), aplicaremos una reducción coherente del tamaño del problema para validar la eficacia del algoritmo. Para este diseño, evaluaremos un conjunto de **12 tomas** con un límite estricto de **4 tomas por día** (requiriendo un mínimo de 3 días para su ejecución).

**Estrategia:**
En lugar de generar exhaustivamente todas las combinaciones posibles y validarlas a posteriori (lo cual desperdicia ciclos de CPU en estados inválidos), implementaremos un algoritmo de **Vuelta Atrás (Backtracking) con Poda Temprana**.

1.  **Fase de Setup:** Cargamos la matriz reducida y pre-calculamos las máscaras de bits para comprimir la información de los actores en registros enteros, optimizando el acceso a memoria.
2.  **Exploración Recursiva:** El algoritmo asigna las tomas una por una de forma secuencial, recorriendo un árbol de decisiones.
3.  **Poda en Tiempo de Ejecución (El Filtro):** Antes de intentar asignar una toma a un día, el algoritmo verifica en tiempo constante $O(1)$ si el día de destino ya alcanzó el límite de 4 tomas. Si la capacidad está agotada, el algoritmo "poda" esa rama de exploración inmediatamente. Esto evita el cálculo recursivo de miles de ramificaciones que, por definición, serían soluciones inválidas.
4.  **Cálculo de Costo:** Solo si el algoritmo logra asignar válidamente la totalidad de las 12 tomas (llegando a una "hoja" del árbol de exploración), calcula el costo global utilizando operaciones lógicas `OR` a nivel de bits y actualiza el resultado si se identifica un nuevo óptimo local.


In [1]:
import math
import random

# SETUP DE DATOS Y REDUCCIÓN COHERENTE

# 1. Definimos la matriz completa que viene en el excel del problema
TOMAS_ACTORES = [
    [1,1,1,1,1,0,0,0,0,0], [0,0,1,1,1,0,0,0,0,0], [0,1,0,0,1,0,1,0,0,0],
    [1,1,0,0,0,0,1,1,0,0], [0,1,0,1,0,0,0,1,0,0], [1,1,0,1,1,0,0,0,0,0],
    [1,1,0,1,1,0,0,0,0,0], [1,1,0,0,0,1,0,0,0,0], [1,1,0,1,0,0,0,0,0,0],
    [1,1,0,0,0,1,0,0,1,0], [1,1,1,0,1,0,0,1,0,0], [1,1,1,1,0,1,0,0,0,0],
    [1,0,0,1,1,0,0,0,0,0], [1,0,1,0,0,1,0,0,0,0], [1,1,0,0,0,0,1,0,0,0],
    [0,0,0,1,0,0,0,0,0,1], [1,0,1,0,0,0,0,0,0,0], [0,0,1,0,0,1,0,0,0,0],
    [1,0,1,0,0,0,0,0,0,0], [1,0,1,1,1,0,0,0,0,0], [0,0,0,0,0,1,0,1,0,0],
    [1,1,1,1,0,0,0,0,0,0], [1,0,1,0,0,0,0,0,0,0], [0,0,1,0,0,1,0,0,0,0],
    [1,1,0,1,0,0,0,0,0,1], [1,0,1,0,1,0,0,0,1,0], [0,0,0,1,1,0,0,0,0,0],
    [1,0,0,1,0,0,0,0,0,0], [1,0,0,0,1,1,0,0,0,0], [1,0,0,1,0,0,0,0,0,0],
]
# Constantes de configuración 
TAMANO_MUESTRA = 12       # Número de tomas a evaluar
TOPE_DIARIO = 4           # Máximo de tomas permitidas por día

lista_tomas_actores = TOMAS_ACTORES[:TAMANO_MUESTRA] 
total_tomas = len(lista_tomas_actores)
total_dias = math.ceil(total_tomas / TOPE_DIARIO)

# Creamos la máscara de bits (dataset reducido de 12 tomas)
mascaras_reducidas = [
    sum(1 << id_actor for id_actor, participa in enumerate(registro) if participa)
    for registro in lista_tomas_actores
]

# Inicializamos las estructuras vacías
asignacion_inicial = [-1] * total_tomas
conteo_inicial = [0] * total_dias

In [2]:
# FUNCIÓN OBJETIVO (Cálculo rápido con bits)

def calcular_costo(asignacion, num_dias, mascaras):

    """
    Calcula el coste operativo total (número de desplazamientos) de una asignación.
    
    Args:
        asignacion (list): Vector donde el índice representa el ID de la toma 
                           y el valor representa el día asignado.
        num_dias (int): Número total de días disponibles en el calendario.
        
    Returns:
        int: Total de actores únicos requeridos en todos los días programados.
    """
    
    # Creamos una lista de ceros, uno por cada día disponible
    actores_por_dia = [0] * num_dias
    
    for toma_id, dia_asignado in enumerate(asignacion):
        # Operación OR (|) para combinar los actores de esta toma con los que ya estaban en el día
        # Unión eficiente de conjuntos mediante bitwise OR: 
        # Acumula los actores de la toma actual en la máscara del día asignado (Operación O(1))
        actores_por_dia[dia_asignado] |= mascaras[toma_id]
        
    costo_total = 0
    for mascara_dia in actores_por_dia:
        # Contamos los bits encendidos ('1') en el entero resultante
        costo_total += bin(mascara_dia).count('1')
        
    return costo_total

In [3]:
# MOTOR DE FUERZA BRUTA (Backtracking con Poda)

def fuerza_bruta_con_poda(toma_actual, asignacion_actual, conteo_por_dia, total_tomas, total_dias):
    global mejor_costo, mejor_asignacion

    """
    Motor recursivo de backtracking que explora el espacio de soluciones aplicando poda.
    
    Args:
        toma_actual (int): Índice de la toma que se está intentando asignar.
        asignacion_actual (list): Estado actual de la asignación.
        conteo_por_dia (list): Registro de cuántas tomas tiene cada día.
        total_tomas (int): Número total de tomas.
        total_dias (int): Número total de días disponibles.
    """
    
    if toma_actual == total_tomas:
        costo_actual = calcular_costo(asignacion_actual, total_dias, mascaras_reducidas)
        if costo_actual < mejor_costo:
            mejor_costo = costo_actual
            mejor_asignacion = asignacion_actual.copy()
        return

    for dia in range(total_dias):
 
        if conteo_por_dia[dia] < TOPE_DIARIO:
            
            asignacion_actual[toma_actual] = dia
            conteo_por_dia[dia] += 1
            
            # Pasamos solo los argumentos necesarios
            fuerza_bruta_con_poda(toma_actual + 1, asignacion_actual, conteo_por_dia, total_tomas, total_dias)
            
            conteo_por_dia[dia] -= 1
            asignacion_actual[toma_actual] = -1

In [4]:
# EJECUCIÓN FUERZA BRUTA

print(f"Iniciando Fuerza Bruta con Poda para {total_tomas} tomas y máximo {TOPE_DIARIO} por día...")

# Variables globales para guardar el mejor resultado durante la recursividad
mejor_costo = float('inf')
mejor_asignacion = []

# Ejecutamos el algoritmo pasando todos los parámetros
fuerza_bruta_con_poda(0, asignacion_inicial, conteo_inicial, total_tomas, total_dias)

# 3. Imprimimos resultados
print("\n--- RESULTADOS ---")
print(f"Mejor costo encontrado (Minimización de desplazamientos): {mejor_costo}")
print(f"Asignación óptima de tomas a días: {mejor_asignacion}")

for indice, dia in enumerate(mejor_asignacion):
    print(f"Toma {indice + 1:02d} -> Asignada al Día {dia}")

Iniciando Fuerza Bruta con Poda para 12 tomas y máximo 4 por día...

--- RESULTADOS ---
Mejor costo encontrado (Minimización de desplazamientos): 18
Asignación óptima de tomas a días: [0, 0, 1, 1, 0, 1, 1, 2, 2, 2, 0, 2]
Toma 01 -> Asignada al Día 0
Toma 02 -> Asignada al Día 0
Toma 03 -> Asignada al Día 1
Toma 04 -> Asignada al Día 1
Toma 05 -> Asignada al Día 0
Toma 06 -> Asignada al Día 1
Toma 07 -> Asignada al Día 1
Toma 08 -> Asignada al Día 2
Toma 09 -> Asignada al Día 2
Toma 10 -> Asignada al Día 2
Toma 11 -> Asignada al Día 0
Toma 12 -> Asignada al Día 2


### Pregunta 7 — Calcula la complejidad del algoritmo por fuerza bruta

*(Análisis de Complejidad Computacional: Backtracking con Poda)*

El problema de asignación de tomas, al ser resuelto mediante la técnica de Backtracking, se clasifica dentro de los problemas de optimización combinatoria. A continuación, se detalla su comportamiento en términos de eficiencia.

**1. Complejidad Temporal:**  
   La complejidad es de tipo exponencial, definida por:
   
   $$ O(D^N \cdot N) $$

   Donde:
* **$N$**: Número total de tomas a asignar - Representa el costo de computar la función objetivo en cada hoja.
* **$D$**: Número total de días disponibles.
* **$D^N$**: Representa el espacio de búsqueda (árbol de decisión de profundidad N y factor de ramificación D).


**2. Impacto de la Poda (Pruning):**  
   Aunque la cota superior sigue siendo exponencial, la condición: <br>
   'if conteo_por_dia[dia] < TOPE_DIARIO' reduce drásticamente el espacio de estados real, permitiendo que instancias pequeñas (como $N$ = 12) se resuelvan en tiempo real.

**3. Escalabilidad:**  

La naturaleza exponencial del problema implica que el tiempo de ejecución crece drásticamente al aumentar $N$.

| $N$ (Tomas)      | D (días según tope) | Operaciones ($D^N$)              | Estado Práctico |
| :---           | :---                 | :---                              | :---            |
| **12** (tope=4)| 3                    | $3^{12} \approx 5.3 \times 10^5$  | **Manejable**   |
| **30** (tope=6)| 5                    | $5^{30} \approx 9.3 \times 10^{20}$| **Inviable**   |

> **Conclusión:** El algoritmo es óptimo para muestras controladas, pero requiere heurísticas para producción a gran escala.

---
## FASE 3 · Diseño y análisis: Algoritmo mejorado

### Pregunta 8 (*) — Diseña un algoritmo que mejore la complejidad de la fuerza bruta. Argumenta por qué crees que mejora

#### Propuesta: Algoritmo de Recocido Simulado con Poda Integrada

El algoritmo diseñado mejora la complejidad al abandonar la búsqueda exhaustiva del árbol de decisiones (que sufre de explosión combinatoria) en favor de una **metaheurística de búsqueda local estocástica**.

#### Argumentación de la mejora frente a la Fuerza Bruta:

**1. Reducción de la Complejidad Temporal (De Exponencial a Lineal)**
* **Fuerza Bruta:** Su complejidad $O(N \cdot D^N)$ es intratable, ya que intenta visitar todas las combinaciones posibles en el espacio de estados.  
* **Recocido Simulado:** Su complejidad es $O(I \cdot N)$, donde $I$ es el número total de iteraciones (una constante definida por los parámetros de enfriamiento). Al ser lineal respecto al número de tomas ($N$), el tiempo de ejecución no se dispara con el tamaño del problema, permitiendo procesar las 30 tomas en segundos en lugar de requerir tiempos computacionales astronómicos.  

**2. Eficiencia mediante la Poda (Pruning) Integrada**
La mejora fundamental no es solo el algoritmo de búsqueda, sino la poda implícita dentro de nuestra función `generar_vecino`. A diferencia de otros algoritmos que generan una solución y luego la descartan si no es válida, nuestro diseño filtra los movimientos antes de aplicarlos:
* Al verificar `conteo[d] < max_tomas_dia` dentro de `generar_vecino`, el algoritmo **poda las ramas de búsqueda inválidas** antes de realizar el cálculo costoso de la función objetivo.  
* Esto garantiza que el algoritmo solo dedique ciclos de CPU a explorar el espacio de soluciones factibles, eliminando por completo el desperdicio de computar estados que violan las restricciones de la productora.

**3. Escape de Óptimos Locales**
A diferencia de una búsqueda voraz (greedy) simple, el Recocido Simulado utiliza el criterio de Metrópolis ($e^{-\Delta/T}$) para aceptar empeoramientos temporales en el coste. Esto nos permite "saltar" fuera de óptimos locales y explorar regiones del espacio de soluciones que un algoritmo voraz no vería, manteniendo una calidad de solución superior sin renunciar a la eficiencia computacional.  


> **Conclusión:** La combinación de una búsqueda metaheurística lineal con una poda de restricciones activa permite transformar un problema NP-difícil en un problema tratable, obteniendo resultados de alta calidad en tiempos de ejecución polinómicos, lo cual es la única aproximación viable para el dataset completo de 30 tomas.

In [5]:
# MEJORA DEL ALGORITMO (SIMULATED ANNEALING)

def solucion_inicial(n_tomas, max_tomas_dia, d_max):
    """
    Genera un reparto secuencial en bloques como punto de partida.
    Garantiza una solución inicial 100% factible.
    """
    return [min(i // max_tomas_dia, d_max - 1) for i in range(n_tomas)]

def generar_vecino(asignacion, n_dias, max_tomas_dia):
    """
    Genera una solución vecina aplicando PODA para garantizar factibilidad.
    
    Operadores:
    1. Intercambio (50%): Cambia de día dos tomas (siempre válido).
    2. Reubicación (50%): Mueve una toma a un día distinto (Aplica poda de capacidad).
    """
    vecino = asignacion[:]
    n = len(vecino)
    
    if random.random() < 0.5:
        # Operador de Intercambio (Swap)
        i, j = random.sample(range(n), 2)
        vecino[i], vecino[j] = vecino[j], vecino[i]
    else:
        # Operador de Reubicación con PODA
        conteo = [0] * n_dias
        for dia in vecino:
            conteo[dia] += 1
            
        i = random.randrange(n)
        dia_actual = vecino[i]
        
        # PODA: Filtramos y descartamos los días que ya superaron el límite
        destinos_posibles = [d for d in range(n_dias) 
                             if d != dia_actual and conteo[d] < max_tomas_dia]
        
        if destinos_posibles:
            vecino[i] = random.choice(destinos_posibles)
            
    return vecino

def recocido_simulado(mascaras, n_tomas, max_tomas_dia, d_max=None, 
                      t_inicial=5.0, t_final=0.01, enfriamiento=0.95, 
                      iteraciones_por_temp=200):
    """
    Motor de la metaheurística de Recocido Simulado.
    
    Args:
        mascaras (list): Lista de bits representando a los actores por toma.
        n_tomas (int): Número total de tomas.
        max_tomas_dia (int): Restricción de tomas máximas.
        d_max (int): Días máximos a utilizar.
        t_inicial, t_final, enfriamiento, iteraciones: Hiperparámetros del SA.
        
    Returns:
        tuple: (Mejor asignación encontrada, Coste de dicha asignación)
    """
    if d_max is None:
        d_max = math.ceil(n_tomas / max_tomas_dia) + 1

    actual = solucion_inicial(n_tomas, max_tomas_dia, d_max)
    costo_actual = calcular_costo(actual, d_max, mascaras) # Reutilizamos tu función ultra rápida
    mejor, costo_mejor = actual[:], costo_actual

    temperatura = t_inicial
    
    while temperatura > t_final:
        for _ in range(iteraciones_por_temp):
            # 1. Generamos vecino (ya viene validado gracias a la poda)
            candidato = generar_vecino(actual, d_max, max_tomas_dia)
            
            # 2. Calculamos su costo con las máscaras de bits
            costo_candidato = calcular_costo(candidato, d_max, mascaras)
            
            # 3. Criterio de Metrópolis (Aceptación)
            delta = costo_candidato - costo_actual
            if delta < 0 or random.random() < math.exp(-delta / temperatura):
                actual, costo_actual = candidato, costo_candidato
                
                # Actualizamos el óptimo global si es necesario
                if costo_actual < costo_mejor:
                    mejor, costo_mejor = actual[:], costo_actual

        # Enfriamos la temperatura
        temperatura *= enfriamiento

    return mejor, costo_mejor

In [6]:
# EJECUCIÓN Y PRUEBA DEL ALGORITMO MEJORADO

print("Preparando prueba para el dataset COMPLETO de 30 tomas...")

# 1. Actualizamos la variable global de máscaras para incluir las 30 tomas

mascara_actores_por_toma = [
    sum(1 << id_actor for id_actor, participa in enumerate(registro) if participa)
    for registro in TOMAS_ACTORES
]

total_tomas_completo = len(TOMAS_ACTORES) # Debería ser 30
tope_diario = 6 # El problema original pide máximo 6 tomas por día

# 2. Fijamos una semilla aleatoria (seed). 
random.seed(42)

print(f"Ejecutando Recocido Simulado para {total_tomas_completo} tomas con un máximo de {tope_diario} por día...\n")

# 3. Disparamos el algoritmo
mejor_asignacion_sa, mejor_costo_sa = recocido_simulado(
    mascaras=mascara_actores_por_toma, 
    n_tomas=total_tomas_completo, 
    max_tomas_dia=tope_diario
)

# 4. Imprimimos los resultados con formato limpio
print("--- RESULTADOS RECOCIDO SIMULADO ---")
print(f"Mejor costo encontrado (Minimización de desplazamientos): {mejor_costo_sa}")
print(f"Días totales utilizados: {max(mejor_asignacion_sa) + 1}")
print(f"Asignación óptima de tomas a días: {mejor_asignacion_sa}\n")

# Detalle día por día de la asignación final obtenida
for dia_objetivo in range(max(mejor_asignacion_sa) + 1):
    tomas_del_dia = [i + 1 for i, d in enumerate(mejor_asignacion_sa) if d == dia_objetivo]
    if tomas_del_dia:
        print(f"Día {dia_objetivo}: Se graban las tomas {tomas_del_dia}")

Preparando prueba para el dataset COMPLETO de 30 tomas...
Ejecutando Recocido Simulado para 30 tomas con un máximo de 6 por día...

--- RESULTADOS RECOCIDO SIMULADO ---
Mejor costo encontrado (Minimización de desplazamientos): 28
Días totales utilizados: 5
Asignación óptima de tomas a días: [2, 0, 3, 3, 3, 2, 2, 4, 1, 4, 4, 1, 2, 1, 3, 3, 1, 1, 0, 0, 4, 2, 2, 1, 3, 4, 0, 0, 4, 0]

Día 0: Se graban las tomas [2, 19, 20, 27, 28, 30]
Día 1: Se graban las tomas [9, 12, 14, 17, 18, 24]
Día 2: Se graban las tomas [1, 6, 7, 13, 22, 23]
Día 3: Se graban las tomas [3, 4, 5, 15, 16, 25]
Día 4: Se graban las tomas [8, 10, 11, 21, 26, 29]


### Pregunta 9 (*) — Calcula la complejidad del algoritmo mejorado

*(Análisis de Complejidad Computacional: Recocido Simulado con Poda)*

A diferencia de la Fuerza Bruta, la complejidad de esta metaheurística no depende de explorar todo el espacio de combinaciones, sino del número de iteraciones definidas por los parámetros de enfriamiento y del costo de evaluar cada estado individual.

#### 1. Complejidad Temporal: $O(N)$ (Lineal)
Para calcular el tiempo de ejecución, desglosamos el algoritmo en sus componentes principales:

* **Ciclo de Temperatura ($K$ iteraciones):** El número de reducciones de temperatura está dado por $K = \log_{\alpha}(T_{final} / T_{inicial})$. Al usar parámetros fijos (ej. $5.0$ a $0.01$ con factor $0.95$), $K$ es una constante independiente del tamaño del problema.
* **Iteraciones por Temperatura ($I_T$):** Se ejecutan 200 iteraciones fijas en cada paso térmico. Esto es otra constante independiente.
* **Operaciones Internas por Iteración:** En cada una de las iteraciones totales ($I = K \cdot I_T$), el algoritmo realiza dos operaciones dependientes de $N$ (donde $N$ es el número de tomas):
  * **Generar un vecino (`generar_vecino`):** Copiar el arreglo de tomas y contar las asignaciones por día requiere $O(N)$ operaciones.
  * **Calcular el costo (`calcular_costo`):** Iterar sobre las $N$ tomas aplicando las máscaras de bits requiere $O(N)$ operaciones.

**Cálculo final del tiempo:**
El tiempo total está dominado por las iteraciones constantes multiplicadas por el trabajo lineal en cada paso:
$$ T(N) = I \cdot ( O(N) + O(N) ) = O(I \cdot N) $$
Dado que $I$ es una constante que no escala con las tomas, la complejidad temporal asintótica se reduce a **$O(N)$**, es decir, un tiempo de ejecución **Lineal**.



#### 2. Complejidad Espacial: $O(N)$
El consumo de memoria de este algoritmo es mínimo y altamente eficiente. A diferencia de las búsquedas exhaustivas completas, aquí no se almacena un árbol de decisiones ni el historial de estados visitados.

En cualquier momento dado, el programa solo mantiene en la memoria RAM:
* El vector de la solución `actual` de longitud $N$ ($O(N)$).
* El vector de la solución `candidato` de longitud $N$ ($O(N)$).
* El vector del `mejor` estado histórico de longitud $N$ ($O(N)$).
* Arreglos auxiliares menores como el conteo de tomas por día ($O(D)$ donde $D \le N$).

**Cálculo final de memoria:**
$$ E(N) = O(N) + O(N) + O(N) + O(D) = O(N) $$


>  **Conclusión:** El algoritmo diseñado logró reducir la complejidad temporal de la clase exponencial inmanejable $O(D^N \cdot N)$ a la clase lineal **$O(N)$**. Esto justifica matemáticamente por qué el dataset completo de 30 tomas se resuelve en fracciones de segundo, garantizando una alta escalabilidad con una huella de memoria estrictamente controlada.

---
## FASE 4 · Validación empírica con datos aleatorios

### Pregunta 10 — Diseña un juego de datos de entrada aleatorio

#### Diseño del Juego de Datos Aleatorios

Para evaluar la capacidad de generalización del algoritmo, se diseña un generador estocástico de matrices de tomas y actores. Este diseño respeta las dimensiones originales del problema ($30$ tomas $\times$ $10$ actores) pero introduce variabilidad realista.

**Lógica del generador:**
1. **Probabilidad de participación:** Se utiliza una probabilidad base (ej. `0.25` o $25\%$) para decidir si un actor participa o no en una toma particular, simulando el comportamiento típico de un guion de cine donde no todos los personajes aparecen constantemente.
2. **Control de validez (Edge Cases):** Dado que se genera de forma aleatoria, existe la posibilidad estadística de generar una fila de ceros (una toma sin ningún actor asignado). El algoritmo detecta esta anomalía y fuerza la inclusión de al menos un actor aleatorio en esa toma, garantizando que el set de datos generado tenga sentido y coherencia con el modelo de negocio.
3. **Conversión de la estructura:** El nuevo dataset aleatorio se transforma inmediatamente a máscaras de bits para mantener la compatibilidad con el motor de cálculo de alta eficiencia.

In [7]:
# GENERACIÓN DE DATOS ALEATORIOS Y APLICACIÓN DEL ALGORITMO

def es_valida(asignacion, max_tomas_dia):
    """Verifica que ningún día supere el límite estricto de tomas permitidas."""
    conteo_por_dia = {}
    for dia in asignacion:
        conteo_por_dia[dia] = conteo_por_dia.get(dia, 0) + 1
    return all(conteo <= max_tomas_dia for conteo in conteo_por_dia.values())

def generar_matriz_aleatoria(n_tomas=30, n_actores=10, prob_participacion=0.25, semilla=None):
    """
    Genera una matriz binaria tomas x actores aleatoria, garantizando que
    ninguna toma se quede sin actores asignados.
    """
    rng = random.Random(semilla)
    matriz = []
    
    for _ in range(n_tomas):
        # Genera 1s y 0s basados en la probabilidad de participación
        fila = [1 if rng.random() < prob_participacion else 0 for _ in range(n_actores)]
        
        # Control de calidad: Si una toma se generó vacía, forzamos un actor al azar
        if sum(fila) == 0:
            fila[rng.randrange(n_actores)] = 1
            
        matriz.append(fila)
        
    return matriz

# Generamos los datos aleatorios
print("--- GENERANDO NUEVO DATASET ---")
matriz_aleatoria = generar_matriz_aleatoria(semilla=99)

# 2. Convertimos la matriz a máscaras de bits para máxima eficiencia
mascaras_aleatorias = [
    sum(1 << a for a, participa in enumerate(fila) if participa)
    for fila in matriz_aleatoria
]

print(f"Nueva matriz generada: {len(matriz_aleatoria)} tomas x {len(matriz_aleatoria[0])} actores")
print(f"Actores por toma (media): {sum(sum(f) for f in matriz_aleatoria) / len(matriz_aleatoria):.2f}\n")

--- GENERANDO NUEVO DATASET ---
Nueva matriz generada: 30 tomas x 10 actores
Actores por toma (media): 2.60



### Pregunta 11 — Aplicacion del algoritmo al juego de datos generado

In [8]:
# APLICACIÓN DEL ALGORITMO DE RECOCIDO SIMULADO CON PODA Y DATOS ALEATORIOS

print("--- APLICANDO RECOCIDO SIMULADO CON PODA ---")
tope_diario_aleatorio = 6
random.seed(11) # Semilla de control para reproducibilidad

mejor_asignacion_aleatoria, mejor_costo_aleatorio = recocido_simulado(
    mascaras=mascaras_aleatorias, 
    n_tomas=len(matriz_aleatoria), 
    max_tomas_dia=tope_diario_aleatorio
)

# 4. Imprimimos el veredicto
print(f"¿Es válida la asignación (respeta tope diario)?: {es_valida(mejor_asignacion_aleatoria, tope_diario_aleatorio)}")
print(f"Días totales utilizados: {max(mejor_asignacion_aleatoria) + 1}")
print(f"Coste obtenido (Minimización): {mejor_costo_aleatorio} desplazamientos-actor")

--- APLICANDO RECOCIDO SIMULADO CON PODA ---
¿Es válida la asignación (respeta tope diario)?: True
Días totales utilizados: 6
Coste obtenido (Minimización): 28 desplazamientos-actor


In [9]:
print("--- VALIDACIÓN CON MÚLTIPLES DATASETS ALEATORIOS ---\n")

escenarios = [
    {"nombre": "Disperso (poca coincidencia de actores)", "n_tomas": 30, "n_actores": 10, "prob": 0.15, "semilla": 7},
    {"nombre": "Denso (muchas coincidencias)",             "n_tomas": 30, "n_actores": 10, "prob": 0.45, "semilla": 21},
    {"nombre": "Escenario ampliado (más tomas y actores)", "n_tomas": 40, "n_actores": 12, "prob": 0.25, "semilla": 33},
]

for esc in escenarios:
    matriz = generar_matriz_aleatoria(
        n_tomas=esc["n_tomas"], n_actores=esc["n_actores"],
        prob_participacion=esc["prob"], semilla=esc["semilla"]
    )
    mascaras = [sum(1 << a for a, participa in enumerate(fila) if participa) for fila in matriz]
    tope = 6
    mejor_asig, mejor_costo = recocido_simulado(mascaras=mascaras, n_tomas=len(matriz), max_tomas_dia=tope)

    print(f"Escenario: {esc['nombre']}")
    print(f"  Tomas={esc['n_tomas']}, Actores={esc['n_actores']}, prob={esc['prob']}, semilla={esc['semilla']}")
    print(f"  ¿Asignación válida?: {es_valida(mejor_asig, tope)}")
    print(f"  Días utilizados: {max(mejor_asig) + 1}")
    print(f"  Coste obtenido: {mejor_costo} desplazamientos-actor\n")

--- VALIDACIÓN CON MÚLTIPLES DATASETS ALEATORIOS ---

Escenario: Disperso (poca coincidencia de actores)
  Tomas=30, Actores=10, prob=0.15, semilla=7
  ¿Asignación válida?: True
  Días utilizados: 6
  Coste obtenido: 25 desplazamientos-actor

Escenario: Denso (muchas coincidencias)
  Tomas=30, Actores=10, prob=0.45, semilla=21
  ¿Asignación válida?: True
  Días utilizados: 6
  Coste obtenido: 38 desplazamientos-actor

Escenario: Escenario ampliado (más tomas y actores)
  Tomas=40, Actores=12, prob=0.25, semilla=33
  ¿Asignación válida?: True
  Días utilizados: 8
  Coste obtenido: 42 desplazamientos-actor




### Pregunta 12. Referencias utilizadas

* **Material principal:** Material de la asignatura *Algoritmos de Optimización* (VIU, Máster en Inteligencia Artificial).
* **Documentación técnica:** Documentación oficial de Python 3.x (Librerías estándar: `random` y `math`), consultada para la implementación estocástica, operaciones a nivel de bits y cálculo de la función objetivo.
* **Fundamentos de Optimización Computacional:** Literatura y conceptos generales sobre optimización combinatoria, diseño de algoritmos de Vuelta Atrás (Backtracking con Poda) y metaheurísticas de búsqueda local (Recocido Simulado / Simulated Annealing).
* **Teoría para Trabajo Futuro:** Conceptos académicos sobre Algoritmos Evolutivos (Genéticos y Meméticos), formulación de problemas clásicos de capacidad (Knapsack Problem) y técnicas de Programación Lineal Entera Mixta (MILP) aplicadas en *solvers* de optimización exacta.


### Pregunta 13. Líneas de trabajo futuro y variaciones del problema

Para avanzar en el estudio de este problema de planificación y acercar el modelo matemático a las necesidades reales de una productora audiovisual, se proponen las siguientes vías de desarrollo:

**1. Escalabilidad y Metaheurísticas Poblacionales (Variaciones al alza del tamaño)**
* El algoritmo de Recocido Simulado actual es muy eficiente para 30 tomas. Sin embargo, si escalamos el problema a la temporada completa de una serie (ej. 500 tomas y 50 actores), la búsqueda basada en una única trayectoria podría perder eficacia. 
* **Propuesta:** Evolucionar hacia **Algoritmos Genéticos (AG)** o **Algoritmos Meméticos** (que hibridan la exploración de un algoritmo genético con la explotación local del recocido simulado). Esto permitiría evaluar múltiples regiones del espacio de soluciones en paralelo.

**2. Variaciones en la Función de Coste (Caché asimétrico)**
* El modelo base asume que todos los actores tienen el mismo salario. En la realidad de la industria, las tarifas varían enormemente entre protagonistas y extras.
* **Propuesta:** Modificar la función objetivo ponderándola mediante un vector de pesos $W$ (salarios). La estructura de máscaras de bits seguiría siendo válida, pero se aplicaría una operación de enmascaramiento lógico para sumar tarifas individuales en lugar de un conteo plano de actores.

**3. Incorporación de Restricciones Físicas y Temporales (Variaciones del problema)**
* **Duración de las tomas:** En lugar de limitar rígidamente la jornada a "6 tomas", se podría asignar un valor de duración (en minutos) a cada toma, cambiando la restricción diaria a un tope de horas laborales (ej. 8 horas/día). Esto transformaría la restricción en un problema análogo al de la *Mochila* (Knapsack).
* **Agendas cruzadas:** Implementar restricciones "duras" de disponibilidad, obligando al algoritmo a respetar calendarios externos donde ciertos actores solo puedan grabar en días específicos.
* **Minimización de huecos (Idle time):** Modificar la evaluación para penalizar agendas fragmentadas, favoreciendo que un actor grabe todas sus escenas en días consecutivos en lugar de alternar días de trabajo y descanso.

**4. Validación mediante Solvers Exactos (MILP)**
* Como línea de investigación teórica, resultaría de gran interés modelar el problema mediante **Programación Lineal Entera Mixta (MILP)**. Aunque la Fuerza Bruta no es viable, utilizar *solvers* comerciales avanzados (como Gurobi o CPLEX) permitiría encontrar el óptimo matemático comprobado para tamaños medianos, sirviendo como "Gold Standard" para medir la desviación exacta de nuestras metaheurísticas.